# AM_AlphaGoZero — Colab T4 setup + TSP-50 K=25 validation pass

Two-part notebook. **Section 1 (one-time setup)** mounts Drive, clones the repo, builds the C++ MCTS extension, and logs into W&B. **Section 2 (validation pass)** re-runs `eval_tsp50_1000_K25.py` on this T4 and diffs the per-instance MCTS column against the reference CSV produced on the local RTX 4060 (`_progress/eval_logs/tsp50_1000_K25_seed1234.csv`).

**Before running:** upload the Stage 1 TSP-50 checkpoint to Drive at

```
MyDrive/AM_AlphaGoZero/checkpoints/stage1_tsp50_am_baseline/epoch-99.pt
```

(local path: `outputs/tsp_50/stage1_tsp50_am_baseline_20260424T032356/epoch-99.pt`, ~5 MB).

**Runtime expectation (T4):** validation pass should finish in ~6–10 min (RTX 4060 reference is 111.8 s on the same args). Per-instance MCTS costs are NOT expected to bit-match across GPU types because cuBLAS picks different matmul kernels — pass criterion is **mean within ±0.001 of the reference 5.74880** (well under the SE of 0.00839).

## Section 1 — One-time setup

### 1.1 Verify GPU + Python + CUDA

In [ ]:
import sys, platform
print('python    =', sys.version.split()[0], platform.platform())
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch     =', torch.__version__, '  cuda available =', torch.cuda.is_available())
print('cuda      =', torch.version.cuda, '  device =', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

### 1.2 Mount Drive + define workspace paths

`WORKSPACE` is where the repo, checkpoints, and outputs live across sessions. Default is `MyDrive/AM_AlphaGoZero`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORKSPACE = '/content/drive/MyDrive/AM_AlphaGoZero'
REPO_DIR = os.path.join(WORKSPACE, 'repo')
CKPT_DIR = os.path.join(WORKSPACE, 'checkpoints')
OUTPUT_DIR = os.path.join(WORKSPACE, 'outputs')

os.makedirs(WORKSPACE, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('WORKSPACE =', WORKSPACE)
print('REPO_DIR  =', REPO_DIR)
print('CKPT_DIR  =', CKPT_DIR)
print('OUTPUT_DIR=', OUTPUT_DIR)

### 1.3 Clone or update repo

Clones to Drive on first run; subsequent runs `git pull` to pick up local commits. Repo is public so no auth is required.

In [ ]:
REPO_URL = 'https://github.com/LejunZhou/AM_ALPHAGOZERO.git'

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned; pulling latest...')
    !git -C {REPO_DIR} pull --ff-only

!git -C {REPO_DIR} log --oneline -1

### 1.4 Install package + build C++ MCTS extension

`--no-deps` keeps Colab's pre-installed torch (CUDA-enabled). Runtime deps installed separately. The C++ extension is plain pybind11 (no torch C++ ABI), so it builds against whatever g++ Colab ships.

In [ ]:
%cd {REPO_DIR}
!pip install --quiet pybind11
!pip install --quiet --no-deps -e .
!pip install --quiet matplotlib numpy scipy tqdm 'wandb>=0.18.0'

In [ ]:
# Verify C++ extension imports cleanly.
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
from am_baseline.search.mcts_cpp import _mcts_cpp as _ext  # noqa: F401
from am_baseline.search.mcts_cpp.solver import CppBatchMCTSSolver  # noqa: F401
print('OK — C++ MCTS extension imports')

### 1.5 W&B login (optional — only needed for training runs)

Add your W&B API key to Colab Secrets as `WANDB_API_KEY` (sidebar → key icon → +). Validation pass doesn't need this; future training runs will.

In [ ]:
try:
    from google.colab import userdata
    key = userdata.get('WANDB_API_KEY')
    import wandb
    wandb.login(key=key)
    print('W&B login OK')
except Exception as e:
    print('Skipping W&B login:', type(e).__name__, str(e)[:120])
    print('(Validation pass does not need W&B; add the secret when you start training runs.)')

### 1.6 Smoke test — quick MCTS sanity on TSP-8

Runs the Phase A smoke battery on a tiny instance to confirm the install end-to-end.

In [ ]:
!cd {REPO_DIR} && PYTHONPATH=src python src/scripts/smoke_mcts.py --backend cpp_batch 2>&1 | tail -20

## Section 2 — Validation pass: TSP-50 K=25 rollout vs reference CSV

### 2.1 Pre-flight — confirm Stage 1 TSP-50 checkpoint is on Drive

In [ ]:
CKPT = os.path.join(CKPT_DIR, 'stage1_tsp50_am_baseline', 'epoch-99.pt')
if not os.path.isfile(CKPT):
    raise FileNotFoundError(
        f'Missing checkpoint at {CKPT}.\n'
        f'Upload outputs/tsp_50/stage1_tsp50_am_baseline_20260424T032356/epoch-99.pt '
        f'from your local repo to that Drive path before running the validation pass.'
    )
print('Checkpoint found:', CKPT, '(', os.path.getsize(CKPT) // 1024, 'KB)')

### 2.2 Run the validation

Identical args to the reference run except `--no_gurobi --no_sample` (skip ~1h of CPU Gurobi solves + AM sampling — we'll compare against the columns already present in the reference CSV). Output CSV goes to Drive for persistence.

In [ ]:
OUT_CSV = os.path.join(OUTPUT_DIR, 'eval_logs', 'tsp50_1000_K25_seed1234_colabT4.csv')
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

!cd {REPO_DIR} && PYTHONPATH=src python src/scripts/eval_tsp50_1000_K25.py \
    --am_ckpt {CKPT} \
    --graph_size 50 \
    --num_test 1000 \
    --seed 1234 \
    --K 25 \
    --mcts_batch_size 64 \
    --no_gurobi \
    --no_sample \
    --out_csv {OUT_CSV}

### 2.3 Diff against the reference CSV

Validation criteria:

| metric | pass threshold | reason |
|---|---|---|
| `AM greedy` mean delta | `< 1e-3` | greedy decode is deterministic per-GPU; cuBLAS reductions may differ across GPU types |
| `MCTS K=25 rollout` mean | within `±0.001` of `5.74880` | well below SE=0.00839; per-instance won't match due to FP nondeterminism propagating through the MCTS tree |
| `MCTS K=25 rollout` per-instance correlation | `> 0.95` | confirms the search is finding *roughly* the same tours, not unrelated ones |

Local sanity check on RTX 4060 (same GPU class as reference): both rows print `PASS` with `max|Δ| = 0.0` (bit-identical).

In [ ]:
import csv
import numpy as np

def load_csv(path):
    with open(path, newline='') as f:
        rows = list(csv.DictReader(f))
    cols = list(rows[0].keys())
    data = {c: np.array([float(r[c]) for r in rows]) for c in cols if c != 'instance'}
    return cols, data, len(rows)

REF_CSV = os.path.join(REPO_DIR, '_progress', 'eval_logs', 'tsp50_1000_K25_seed1234.csv')
ref_cols, ref, n_ref = load_csv(REF_CSV)
new_cols, new, n_new = load_csv(OUT_CSV)

print(f'reference rows = {n_ref}, colab rows = {n_new}')
print(f'reference cols = {ref_cols}')
print(f'colab cols     = {new_cols}')

def diff_col(name, ref_col=None, ref_mean_target=None, mean_tol=1e-3, corr_tol=0.95):
    ref_col = ref_col or name
    if name not in new:
        print(f'  [skip] {name} not in colab CSV')
        return
    r = ref[ref_col]
    n = new[name]
    d = n - r
    mean_n = float(n.mean())
    target = ref_mean_target if ref_mean_target is not None else float(r.mean())
    corr = float(np.corrcoef(r, n)[0, 1])
    ok_mean = abs(mean_n - target) < mean_tol
    ok_corr = corr > corr_tol
    flag = 'PASS' if (ok_mean and ok_corr) else 'FAIL'
    print(f'  [{flag}] {name}:')
    print(f'        ref mean = {r.mean():.5f}   colab mean = {mean_n:.5f}   |Δmean| = {abs(mean_n - target):.6f}  (tol {mean_tol})')
    print(f'        Δ per-instance:  mean = {d.mean():+.6f}  max|Δ| = {np.abs(d).max():.6f}  std = {d.std():.6f}')
    print(f'        Pearson corr (ref vs colab) = {corr:.6f}  (tol > {corr_tol})')

print('\n--- Validation report ---')
diff_col('AM greedy', mean_tol=1e-3, corr_tol=0.99)
diff_col('MCTS K=25 rollout', ref_mean_target=5.74880, mean_tol=1e-3, corr_tol=0.95)

### 2.4 Recap

If both rows print `PASS`, the Colab T4 environment is good for any of the cheap probe scripts:

- `val_stage4_mcts.py` — leaf-eval bypass sweeps on any new Stage 4 checkpoint
- `probe_value_aleatoric.py`, `probe_grad_norm.py`, `probe_mcts_quality.py`
- `eval_tsp{20,50}_full_comparison.py` (full comparison tables)
- 20-iter K-comparison ablations, sub-budget probes
- `probe_mcts_decomp.py` — wall-time decomposition (T4 numbers, tagged separately)

Heavy lifting (TSP-50 lv0 step-decay 400-iter, TSP-100 from-scratch) stays on Modal A10G so wall-time numbers in `_progress/stage5_progress.md` remain comparable.

**Next session pattern:** re-run cells 1.1–1.4 (fast, ~1 min — cached install) + 1.5 if you need W&B, then jump straight to whatever probe is on the docket.